In [ ]:
import os, re, glob, random, warnings, hashlib
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report
from lightgbm import LGBMClassifier, early_stopping

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False

warnings.filterwarnings("ignore")

for old_f in ["submission.csv", "/kaggle/working/submission.csv"]:
    if os.path.exists(old_f):
        os.remove(old_f)

# =====================================================================
# 0. CONFIGURATION & REPRODUCIBILITY
# =====================================================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running V20-NeuralFocus (3 Deep Nets + Cross-Subject SupConLoss) on: {DEVICE}")

EMG_COLS = [f'ch{i}' for i in range(1, 9)]
IMU_COLS = ['ax', 'ay', 'az', 'gx', 'gy', 'gz']
SIGNAL_COLS = EMG_COLS + IMU_COLS
GESTURES = ['rest', 'fist', 'open', 'point', 'pinch', 'thumb-up']
GESTURE_MAP = {g: i for i, g in enumerate(GESTURES)}
INV_GESTURE_MAP = {i: g for i, g in enumerate(GESTURES)}

THETA_8 = np.linspace(0, 2 * np.pi, 8, endpoint=False).astype(np.float32)
COS_8, SIN_8 = np.cos(THETA_8), np.sin(THETA_8)

SEQ_LEN = 192
WINDOW_SEC = 1.4
TRAIN_STRIDE_SEC = 0.40
BATCH_SIZE = 64
EPOCHS = 36
LR = 1.1e-3
N_FOLDS = 5

DATA_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if any(f.startswith('train_') and f.endswith('.xlsx') for f in files):
        DATA_DIR = os.path.dirname(root) if os.path.basename(root) == 'train' else root
        break
if DATA_DIR is None:
    DATA_DIR = '/kaggle/input/competitions/vr-calibration-free-emg-challenge-pieeg'

def natural_key(path):
    nums = re.findall(r'\d+', os.path.basename(path))
    return int(nums[0]) if nums else 0

def read_aura_excel_fast(fpath):
    xl = pd.ExcelFile(fpath, engine='openpyxl')
    sheet = 'multimodal' if 'multimodal' in xl.sheet_names else xl.sheet_names[0]
    df = xl.parse(sheet)
    df.columns = [str(c).replace('#', '').strip() for c in df.columns]
    return fpath, df

# =====================================================================
# 1. CLEAN SIGNAL DECOMPOSITION & TWO-PASS IMPEDANCE CALIBRATION
# =====================================================================
def decompose_window_signals(df_win):
    raw = df_win[SIGNAL_COLS].astype(np.float32).values.copy()
    T = raw.shape[0]

    # Interpolate frozen imu_new == 0 rows into smooth continuous trajectories
    if 'imu_new' in df_win.columns and T > 6:
        imu_new_mask = (pd.to_numeric(df_win['imu_new'], errors='coerce').fillna(1).values == 1)
        if np.sum(imu_new_mask) >= 4:
            raw[~imu_new_mask, 8:] = np.nan

    raw = pd.DataFrame(raw).interpolate(method='linear', limit_direction='both').fillna(0.0).values
    centered = raw - np.mean(raw, axis=0, keepdims=True)

    if T > 4:
        t_axis = np.linspace(-1.0, 1.0, T, dtype=np.float32)[:, None]
        slope = np.sum(centered * t_axis, axis=0, keepdims=True) / (np.sum(t_axis ** 2) + 1e-8)
        detrended = centered - slope * t_axis
        slope_vec = slope[0].astype(np.float32)
    else:
        detrended = centered
        slope_vec = np.zeros(14, dtype=np.float32)

    emg_det = detrended[:, :8]
    k_w = min(21, max(5, T // 10))
    emg_slow = pd.DataFrame(emg_det).rolling(window=k_w, min_periods=1, center=True).mean().values.astype(np.float32)
    emg_fast = (emg_det - emg_slow).astype(np.float32)

    imu_full = pd.DataFrame(centered[:, 8:]).rolling(window=5, min_periods=1, center=True).mean().values.astype(np.float32)
    imu_full = imu_full - np.mean(imu_full, axis=0, keepdims=True)

    imu_det = pd.DataFrame(detrended[:, 8:]).rolling(window=5, min_periods=1, center=True).mean().values.astype(np.float32)
    imu_det = imu_det - np.mean(imu_det, axis=0, keepdims=True)

    sig_fast = np.concatenate([emg_fast, imu_det], axis=1).astype(np.float32)
    return sig_fast, emg_slow, imu_full, slope_vec

def compute_session_channel_gains(window_dfs):
    rms_list = []
    for df_w in window_dfs:
        sig_fast, _, _, _ = decompose_window_signals(df_w)
        rms_list.append(np.sqrt(np.mean(sig_fast ** 2, axis=0)) + 1e-5)
    rms_mat = np.stack(rms_list)
    return (np.percentile(rms_mat, 60, axis=0) + 1e-4).astype(np.float32)

def extract_v20_features(df_win, session_ch_gain, seq_len=SEQ_LEN):
    sig_uncal, emg_slow_u, imu_full_u, slope_u = decompose_window_signals(df_win)
    T = sig_uncal.shape[0]

    sig_cal = sig_uncal / session_ch_gain[None, :]
    emg_cal = sig_cal[:, :8]
    imu_cal = sig_cal[:, 8:]
    emg_slow_cal = emg_slow_u / session_ch_gain[:8][None, :]
    imu_full_cal = imu_full_u / session_ch_gain[8:][None, :]
    slope_cal = slope_u / session_ch_gain

    # Stream A: V1 36-Ch Per-Window Z-Scored Waveform
    delta_uncal = np.diff(sig_uncal, axis=0, prepend=sig_uncal[:1, :])
    emg_u = sig_uncal[:, :8]
    emg_p = np.roll(emg_u, 1, axis=0); emg_p[0] = emg_u[0]
    emg_n = np.roll(emg_u, -1, axis=0); emg_n[-1] = emg_u[-1]
    tkeo_u = (emg_u ** 2) - (emg_p * emg_n)

    v1_raw = np.concatenate([sig_uncal, delta_uncal, tkeo_u], axis=1)
    old_idx, new_idx = np.linspace(0.0, 1.0, T), np.linspace(0.0, 1.0, seq_len)
    v1_res = np.column_stack([np.interp(new_idx, old_idx, v1_raw[:, c]) for c in range(36)]).astype(np.float32)
    v1_norm = np.clip((v1_res - np.mean(v1_res, axis=0, keepdims=True)) / (np.std(v1_res, axis=0, keepdims=True) + 1e-5), -8.0, 8.0)

    # Stream B: V3 30-Ch Impedance-Calibrated Spatial & Envelope Stream
    env_k = max(5, T // 14)
    emg_env_cal = np.sqrt(pd.DataFrame(emg_cal ** 2).rolling(window=env_k, min_periods=1, center=True).mean().values + 1e-6)
    spatial_prop_cal = emg_env_cal / (np.sum(emg_env_cal, axis=1, keepdims=True) + 1e-4)
    v3_raw = np.concatenate([
        np.sign(emg_cal) * np.log1p(np.abs(emg_cal)),
        np.log1p(emg_env_cal),
        spatial_prop_cal,
        np.sign(imu_full_cal) * np.log1p(np.abs(imu_full_cal))
    ], axis=1)
    v3_res = np.column_stack([np.interp(new_idx, old_idx, v3_raw[:, c]) for c in range(30)]).astype(np.float32)
    seq_tensor = np.concatenate([v1_norm, np.clip(v3_res, -8.0, 8.0)], axis=1).T.astype(np.float32)

    # Part C: Noise-Free Biomechanical Features
    feats = []
    emg_diff_cal = np.diff(emg_cal, axis=0, prepend=emg_cal[:1, :])
    rms_cal = np.sqrt(np.mean(emg_cal ** 2, axis=0)) + 1e-5
    mav_cal = np.mean(np.abs(emg_cal), axis=0) + 1e-5
    wl_cal = np.mean(np.abs(emg_diff_cal), axis=0) + 1e-5
    p90_cal = np.percentile(np.abs(emg_cal), 90, axis=0) + 1e-5
    p50_cal = np.percentile(np.abs(emg_cal), 50, axis=0) + 1e-5
    zc_cal = np.sum(np.diff(np.sign(emg_cal), axis=0) != 0, axis=0) / float(T)

    feats.extend(np.log1p(rms_cal)); feats.extend(np.log1p(mav_cal)); feats.extend(np.log1p(wl_cal))
    feats.extend(np.log1p(p90_cal)); feats.extend(np.log1p(p50_cal)); feats.extend(zc_cal)

    rms_slow = np.sqrt(np.mean(emg_slow_cal ** 2, axis=0)) + 1e-5
    feats.extend(np.log1p(rms_slow))
    feats.extend(rms_slow / np.sum(rms_slow))
    feats.extend(np.sign(slope_cal) * np.log1p(np.abs(slope_cal)))

    rms_prop = rms_cal / np.sum(rms_cal)
    feats.extend(rms_prop); feats.extend(wl_cal / np.sum(wl_cal))
    feats.extend(p90_cal / np.sum(p90_cal)); feats.extend(mav_cal / np.sum(mav_cal))
    rms_u = np.sqrt(np.mean(emg_u ** 2, axis=0)) + 1e-5
    feats.extend(rms_u / np.sum(rms_u))

    tot_rms_cal = float(np.log1p(np.sum(rms_cal)))
    tot_wl_cal = float(np.log1p(np.sum(wl_cal)))
    entropy = float(-np.sum(rms_prop * np.log(rms_prop + 1e-6)))
    cx, cy = float(np.sum(rms_prop * COS_8)), float(np.sum(rms_prop * SIN_8))
    polar_r = float(np.sqrt(cx ** 2 + cy ** 2))
    polar_ang = float(np.arctan2(cy, cx))

    feats.extend([
        tot_rms_cal, tot_wl_cal, entropy, float(np.max(rms_prop)), float(np.min(rms_prop)),
        float(np.std(rms_prop)), float(np.log1p(np.max(rms_cal))), float(np.log1p(np.median(rms_cal))),
        float(np.max(rms_prop) / (np.median(rms_prop) + 1e-4)), float(np.argmax(rms_prop)),
        cx, cy, polar_r, polar_ang
    ])
    feats.extend(np.sort(rms_prop))
    for sh in [1, 2, 3, 4]:
        feats.append(float(np.sum(rms_prop * np.roll(rms_prop, sh))))

    for i in range(8):
        feats.append(float(rms_prop[i] - rms_prop[(i + 4) % 8]))
        feats.append(float(rms_prop[i] - rms_prop[(i + 1) % 8]))
        feats.append(float(np.log1p(rms_cal[i]) - np.log1p(rms_cal[(i + 4) % 8])))

    cov_cal = (emg_cal.T @ emg_cal) / float(T) + 1e-3 * np.eye(8)
    eigvals_c, eigvecs_c = np.linalg.eigh(cov_cal)
    eigvals_c = np.clip(eigvals_c, 1e-4, None)
    log_cov_cal = eigvecs_c @ np.diag(np.log(eigvals_c)) @ eigvecs_c.T
    feats.extend(log_cov_cal[np.triu_indices(8, k=0)])
    feats.extend(np.log(eigvals_c))
    corr_mat = np.nan_to_num(np.corrcoef(emg_cal.T), nan=0.0)
    feats.extend(corr_mat[np.triu_indices(8, k=1)])

    for sub in np.array_split(emg_env_cal, 3, axis=0):
        s_m = np.mean(sub, axis=0) + 1e-5
        feats.extend(np.log1p(s_m)); feats.extend(s_m / np.sum(s_m))
    feats.extend(np.percentile(emg_env_cal, 90, axis=0) - np.percentile(emg_env_cal, 20, axis=0))
    feats.extend(np.mean(emg_env_cal[T // 4 : 3 * T // 4], axis=0) / (np.mean(emg_env_cal[:T // 4], axis=0) + 1e-3))

    fft_p = np.abs(np.fft.rfft(emg_cal, axis=0)) ** 2
    if len(fft_p) > 6:
        b1, b2, b3 = np.array_split(fft_p[1:], 3, axis=0)
        p1, p2, p3 = np.mean(b1, axis=0) + 1e-5, np.mean(b2, axis=0) + 1e-5, np.mean(b3, axis=0) + 1e-5
        ptot = p1 + p2 + p3
        feats.extend(p1 / ptot); feats.extend(p2 / ptot); feats.extend(p3 / ptot)
    else:
        feats.extend(np.zeros(24))

    imu_rms_cal = np.sqrt(np.mean(imu_full_cal ** 2, axis=0)) + 1e-5
    feats.extend(np.log1p(imu_rms_cal))
    feats.extend(np.log1p(np.percentile(np.abs(imu_full_cal), 90, axis=0)))
    feats.extend(np.percentile(imu_full_cal, 10, axis=0))
    feats.extend(np.percentile(imu_full_cal, 90, axis=0))
    feats.extend(imu_rms_cal[:3] / (np.sum(imu_rms_cal[:3]) + 1e-5))
    feats.extend(imu_rms_cal[3:] / (np.sum(imu_rms_cal[3:]) + 1e-5))

    # Non-zero cumulative trajectory excursion (ptp & mean)
    imu_traj = np.cumsum(imu_full_cal, axis=0) / float(T)
    feats.extend(np.log1p(np.ptp(imu_traj, axis=0)))
    feats.extend(np.mean(imu_traj, axis=0))
    feats.extend(np.mean(imu_full_cal[T // 2:], axis=0) - np.mean(imu_full_cal[:T // 2], axis=0))

    acc_mag = np.sqrt(np.sum(imu_full_cal[:, :3] ** 2, axis=1))
    gyro_mag = np.sqrt(np.sum(imu_full_cal[:, 3:] ** 2, axis=1))
    feats.extend([
        float(np.log1p(np.mean(acc_mag))), float(np.log1p(np.std(acc_mag))),
        float(np.log1p(np.mean(gyro_mag))), float(np.log1p(np.std(gyro_mag))),
        float(np.log1p(np.sum(rms_cal)) / (np.log1p(np.mean(gyro_mag)) + 0.1))
    ])
    imu_corr = np.nan_to_num(np.corrcoef(imu_full_cal.T), nan=0.0)
    feats.extend(imu_corr[np.triu_indices(6, k=1)])

    tab_vec = np.nan_to_num(np.array(feats, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    return seq_tensor, tab_vec

def append_session_aligned_features(X_tab, session_ids):
    N, D = X_tab.shape
    X_sess_z = np.zeros_like(X_tab)
    X_sess_rank = np.zeros((N, 64), dtype=np.float32)

    for s_id in np.unique(session_ids):
        mask = (session_ids == s_id)
        if np.sum(mask) < 3:
            continue
        sub = X_tab[mask]
        med = np.median(sub, axis=0, keepdims=True)
        iqr = (np.percentile(sub, 75, axis=0, keepdims=True) - np.percentile(sub, 25, axis=0, keepdims=True)) + 1e-4
        X_sess_z[mask] = np.clip((sub - med) / iqr, -6.0, 6.0)

        for c in range(64):
            vals = sub[:, c]
            X_sess_rank[mask, c] = (np.argsort(np.argsort(vals)).astype(np.float32) / float(max(1, len(vals) - 1))) - 0.5

    return np.concatenate([X_tab, X_sess_z, X_sess_rank], axis=1)

# =====================================================================
# 2. PARALLEL DATASET LOADER (PURE CLASSIFICATION + VARIANCE GUARD)
# =====================================================================
def process_single_train_session(args):
    session_idx, (fpath, df) = args
    if 'label_source' in df.columns:
        df = df[df['label_source'].astype(str).str.strip().str.lower() == 'guided'].reset_index(drop=True)
    label_col = 'label' if 'label' in df.columns else 'gesture'
    df[label_col] = df[label_col].astype(str).str.strip().str.lower()
    df = df[df[label_col].isin(GESTURE_MAP.keys())].reset_index(drop=True)
    if len(df) == 0:
        return None

    df['seg_id'] = ((df['t'].diff().abs() > 0.25) | (df[label_col] != df[label_col].shift())).cumsum()
    raw_windows, cls_list, anchor_list = [], [], []

    for _, seg in df.groupby('seg_id'):
        t_start, t_end = seg['t'].iloc[0], seg['t'].iloc[-1]
        gesture_id = GESTURE_MAP[seg[label_col].iloc[0]]
        curr_t, step_idx = t_start, 0
        while curr_t + WINDOW_SEC <= t_end + 0.08:
            win = seg[(seg['t'] >= curr_t) & (seg['t'] < curr_t + WINDOW_SEC)]
            if len(win) >= 35:
                raw_windows.append(win)
                cls_list.append(gesture_id)
                anchor_list.append(1 if (step_idx % 3 == 0) else 0)
                step_idx += 1
            curr_t += TRAIN_STRIDE_SEC

    if not raw_windows:
        return None
    anchor_wins = [w for w, is_a in zip(raw_windows, anchor_list) if is_a == 1]
    session_ch_gain = compute_session_channel_gains(anchor_wins if len(anchor_wins) >= 5 else raw_windows)
    pairs = [extract_v20_features(win, session_ch_gain, seq_len=SEQ_LEN) for win in raw_windows]
    seq_list, tab_list = zip(*pairs)
    n = len(seq_list)
    return (
        np.stack(seq_list), np.stack(tab_list), np.array(cls_list, dtype=np.int64),
        np.full(n, session_idx, dtype=np.int64), np.array(anchor_list, dtype=np.int64)
    )

def load_v20_datasets(data_dir):
    train_files = sorted(glob.glob(os.path.join(data_dir, "**", "train_*.xlsx"), recursive=True), key=natural_key)
    test_files = sorted(glob.glob(os.path.join(data_dir, "**", "test_*.xlsx"), recursive=True), key=natural_key)
    print(f"Parallel-loading {len(train_files)} train files and {len(test_files)} test files...")

    with ThreadPoolExecutor(max_workers=4) as ex:
        train_loaded = list(ex.map(read_aura_excel_fast, train_files))
        test_loaded = list(ex.map(read_aura_excel_fast, test_files))
        train_res = [r for r in ex.map(process_single_train_session, enumerate(train_loaded)) if r is not None]

    X_seq = np.concatenate([r[0] for r in train_res], axis=0)
    X_tab = np.concatenate([r[1] for r in train_res], axis=0)
    y_cls = np.concatenate([r[2] for r in train_res], axis=0)
    groups = np.concatenate([r[3] for r in train_res], axis=0)
    is_anchor = np.concatenate([r[4] for r in train_res], axis=0)

    X_test_seq, X_test_tab, test_ids, test_sessions = [], [], [], []
    for test_sess_idx, (fpath, df) in enumerate(test_loaded):
        id_col = 'action_id' if 'action_id' in df.columns else df.columns[0]
        sess_wins, sess_ids = [], []
        for act_id, win in df.groupby(id_col, sort=False):
            sess_wins.append(win); sess_ids.append(act_id)
        test_ch_gain = compute_session_channel_gains(sess_wins)
        for act_id, win in zip(sess_ids, sess_wins):
            s_t, t_v = extract_v20_features(win, test_ch_gain, seq_len=SEQ_LEN)
            X_test_seq.append(s_t); X_test_tab.append(t_v)
            test_ids.append(act_id); test_sessions.append(test_sess_idx)

    X_test_seq, X_test_tab = np.stack(X_test_seq), np.stack(X_test_tab)
    test_sessions = np.array(test_sessions, dtype=np.int64)

    X_tab = append_session_aligned_features(X_tab, groups)
    X_test_tab = append_session_aligned_features(X_test_tab, test_sessions)

    # Strip any near-zero variance column
    valid_cols = np.std(X_tab, axis=0) > 1e-4
    X_tab = X_tab[:, valid_cols]
    X_test_tab = X_test_tab[:, valid_cols]

    print(f"Train Windows: {len(X_seq)} (Anchors: {np.sum(is_anchor)}) | Seq: {X_seq.shape} | Clean Tab: {X_tab.shape}")
    return X_seq, X_tab, y_cls, groups, is_anchor, X_test_seq, X_test_tab, test_ids, test_sessions

# =====================================================================
# 3. THREE PURE-CLASSIFICATION NEURAL ARCHITECTURES + SUPCON LOSS
# =====================================================================
class AuraPureClsDataset(Dataset):
    def __init__(self, X_seq, X_tab, y_cls=None, is_train=True):
        self.X_seq, self.X_tab = X_seq, X_tab
        self.y_cls = y_cls
        self.is_train = is_train

    def __len__(self):
        return len(self.X_seq)

    def __getitem__(self, idx):
        x, tab = self.X_seq[idx].copy(), self.X_tab[idx].copy()
        if self.is_train:
            if random.random() < 0.55:
                x = np.roll(x, shift=random.randint(-8, 8), axis=-1)
            if random.random() < 0.45:
                x = x * np.random.uniform(0.90, 1.10, size=(x.shape[0], 1)).astype(np.float32) + np.random.normal(0.0, 0.02, size=x.shape).astype(np.float32)
            if random.random() < 0.25:
                t0 = random.randint(0, x.shape[1] - 16)
                x[:, t0 : t0 + 16] = 0.0
        if self.y_cls is None:
            return torch.tensor(x, dtype=torch.float32), torch.tensor(tab, dtype=torch.float32)
        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(tab, dtype=torch.float32),
            torch.tensor(self.y_cls[idx], dtype=torch.long)
        )

class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(channels, channels // reduction), nn.GELU(), nn.Linear(channels // reduction, channels), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x.mean(dim=-1)).unsqueeze(-1)

class AuraCrossAttnConformer(nn.Module):
    """
    Neural Net #1: Multi-Scale CNN + Conformer where the Tabular Biomechanical Embedding
    dynamically gates and queries the temporal waveform tokens via FiLM & Cross-Attention!
    """
    def __init__(self, in_channels=66, tab_dim=850, d_model=128, num_classes=6):
        super().__init__()
        self.inst_norm_v1 = nn.InstanceNorm1d(36, affine=True)
        self.bn_v3 = nn.BatchNorm1d(in_channels - 36)
        self.conv_k3 = nn.Conv1d(in_channels, d_model // 2, kernel_size=3, padding=1)
        self.conv_k7 = nn.Conv1d(in_channels, d_model // 4, kernel_size=7, padding=3)
        self.conv_k15 = nn.Conv1d(in_channels, d_model // 4, kernel_size=15, padding=7)
        self.cnn_encoder = nn.Sequential(
            nn.BatchNorm1d(d_model), nn.GELU(), SEBlock1D(d_model), nn.Dropout(0.15), nn.MaxPool1d(2, 2),
            nn.Conv1d(d_model, d_model, kernel_size=5, padding=2), nn.BatchNorm1d(d_model), nn.GELU(), SEBlock1D(d_model), nn.MaxPool1d(2, 2)
        )
        self.pos_emb = nn.Parameter(torch.randn(1, 48, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=4, dim_feedforward=d_model * 2, dropout=0.20, activation='gelu', batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=2)
        self.attn_pool = nn.Sequential(nn.Linear(d_model, d_model // 2), nn.Tanh(), nn.Linear(d_model // 2, 1))

        self.tab_mlp = nn.Sequential(
            nn.LayerNorm(tab_dim), nn.Dropout(0.25), nn.Linear(tab_dim, 192), nn.BatchNorm1d(192), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(192, d_model), nn.BatchNorm1d(d_model), nn.GELU()
        )
        # Tabular-to-Waveform FiLM Gate
        self.film_gamma = nn.Linear(d_model, d_model * 2)
        self.film_beta = nn.Linear(d_model, d_model * 2)

        fused_dim = d_model * 3
        self.embed_proj = nn.Sequential(
            nn.LayerNorm(fused_dim), nn.Dropout(0.25), nn.Linear(fused_dim, 128), nn.BatchNorm1d(128), nn.GELU()
        )
        self.supcon_head = nn.Linear(128, 64, bias=False)
        self.cls_head = nn.Linear(128, num_classes)

    def forward(self, x_seq, x_tab):
        x = torch.cat([self.inst_norm_v1(x_seq[:, :36, :]), self.bn_v3(x_seq[:, 36:, :])], dim=1)
        x = self.cnn_encoder(torch.cat([self.conv_k3(x), self.conv_k7(x), self.conv_k15(x)], dim=1))
        seq = self.transformer(x.permute(0, 2, 1) + self.pos_emb)
        w = F.softmax(self.attn_pool(seq), dim=1)
        seq_feat = torch.cat([torch.sum(seq * w, dim=1), torch.max(seq, dim=1)[0]], dim=-1)

        tab_e = self.tab_mlp(x_tab)
        gated_seq = seq_feat * (1.0 + 0.5 * torch.tanh(self.film_gamma(tab_e))) + 0.5 * self.film_beta(tab_e)
        emb = self.embed_proj(torch.cat([gated_seq, tab_e], dim=-1))
        z_norm = F.normalize(self.supcon_head(emb), dim=-1)
        return self.cls_head(emb), z_norm

class InceptionWaveBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        mid = out_c // 4
        self.b1, self.b2 = nn.Conv1d(in_c, mid, 1), nn.Conv1d(in_c, mid, 5, padding=2)
        self.b3, self.b4 = nn.Conv1d(in_c, mid, 11, padding=5), nn.Conv1d(in_c, mid, 23, padding=11)
        self.bn = nn.BatchNorm1d(out_c)
        self.short = nn.Conv1d(in_c, out_c, 1) if in_c != out_c else nn.Identity()
    def forward(self, x):
        return F.gelu(self.bn(torch.cat([self.b1(x), self.b2(x), self.b3(x), self.b4(x)], dim=1)) + self.short(x))

class InceptionWaveBiGRUNet(nn.Module):
    """
    Neural Net #2: Multi-Scale InceptionTime + BiGRU + Attentive Statistics Pooling (No reg_head noise!).
    """
    def __init__(self, in_channels=66, tab_dim=850, num_classes=6):
        super().__init__()
        self.inc1, self.pool1 = InceptionWaveBlock(in_channels, 96), nn.AvgPool1d(2, 2)
        self.inc2, self.pool2 = InceptionWaveBlock(96, 128), nn.AvgPool1d(2, 2)
        self.inc3 = InceptionWaveBlock(128, 128)
        self.bigru = nn.GRU(128, 64, num_layers=2, batch_first=True, bidirectional=True, dropout=0.2)
        self.tab_mlp = nn.Sequential(
            nn.LayerNorm(tab_dim), nn.Dropout(0.25), nn.Linear(tab_dim, 192), nn.BatchNorm1d(192), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(192, 128), nn.BatchNorm1d(128), nn.GELU()
        )
        fused_dim = 128 * 3 + 128
        self.embed_proj = nn.Sequential(
            nn.LayerNorm(fused_dim), nn.Dropout(0.25), nn.Linear(fused_dim, 128), nn.BatchNorm1d(128), nn.GELU()
        )
        self.supcon_head = nn.Linear(128, 64, bias=False)
        self.cls_head = nn.Linear(128, num_classes)

    def forward(self, x_seq, x_tab):
        h = self.inc3(self.pool2(self.inc2(self.pool1(self.inc1(x_seq)))))
        seq_g, _ = self.bigru(h.permute(0, 2, 1))
        h = h + seq_g.permute(0, 2, 1)
        fused = torch.cat([h.mean(dim=-1), h.std(dim=-1), h.max(dim=-1)[0], self.tab_mlp(x_tab)], dim=-1)
        emb = self.embed_proj(fused)
        z_norm = F.normalize(self.supcon_head(emb), dim=-1)
        return self.cls_head(emb), z_norm

class CircularForearmResNet(nn.Module):
    """
    Neural Net #3: Treats the 8 EMG streams as a 2D Circular Ring (8 electrodes x Time) with
    circular wrap-around padding so electrode shifts across unseen users are handled natively!
    """
    def __init__(self, in_channels=66, tab_dim=850, num_classes=6):
        super().__init__()
        # Extract 5 EMG-specific 8-channel groups from the 66 channels:
        # [0:8 raw_v1, 14:22 delta_v1, 28:36 tkeo_v1, 36:44 cal_emg, 44:52 cal_env] -> (B, 5, 8, T)
        self.ring_conv1 = nn.Conv2d(5, 32, kernel_size=(3, 5), stride=(1, 2), padding=(0, 2))
        self.ring_bn1 = nn.BatchNorm2d(32)
        self.ring_conv2 = nn.Conv2d(32, 48, kernel_size=(3, 5), stride=(1, 2), padding=(0, 2))
        self.ring_bn2 = nn.BatchNorm2d(48)

        # Parallel 1D ResNet on full 66-channel multimodal stream
        self.full_stem = nn.Sequential(
            nn.Conv1d(in_channels, 96, kernel_size=5, stride=2, padding=2), nn.BatchNorm1d(96), nn.GELU(),
            SEBlock1D(96),
            nn.Conv1d(96, 128, kernel_size=5, stride=2, padding=2), nn.BatchNorm1d(128), nn.GELU(),
            SEBlock1D(128)
        )
        self.tab_mlp = nn.Sequential(
            nn.LayerNorm(tab_dim), nn.Dropout(0.25), nn.Linear(tab_dim, 192), nn.BatchNorm1d(192), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(192, 128), nn.BatchNorm1d(128), nn.GELU()
        )
        # Ring pooled (48 * 2 = 96) + Full stream pooled (128 * 2 = 256) + Tabular (128) = 480
        self.embed_proj = nn.Sequential(
            nn.LayerNorm(480), nn.Dropout(0.25), nn.Linear(480, 128), nn.BatchNorm1d(128), nn.GELU()
        )
        self.supcon_head = nn.Linear(128, 64, bias=False)
        self.cls_head = nn.Linear(128, num_classes)

    def forward(self, x_seq, x_tab):
        B, _, T = x_seq.shape
        emg_5 = torch.stack([
            x_seq[:, 0:8, :], x_seq[:, 14:22, :], x_seq[:, 28:36, :],
            x_seq[:, 36:44, :], x_seq[:, 44:52, :]
        ], dim=1)  # (B, 5, 8, T)
        # Circular padding around the 8-electrode forearm circumference (ch8 wraps to ch1)
        emg_pad1 = torch.cat([emg_5[:, :, -1:, :], emg_5, emg_5[:, :, :1, :]], dim=2)
        r = F.gelu(self.ring_bn1(self.ring_conv1(emg_pad1)))
        r_pad2 = torch.cat([r[:, :, -1:, :], r, r[:, :, :1, :]], dim=2)
        r = F.gelu(self.ring_bn2(self.ring_conv2(r_pad2)))  # (B, 48, 8, 48)
        # Rotation-invariant pooling across the 8 electrodes (mean & max over electrode ring)
        r_time = 0.5 * (r.mean(dim=2) + r.max(dim=2)[0])   # (B, 48, 48)
        ring_feat = torch.cat([r_time.mean(dim=-1), r_time.max(dim=-1)[0]], dim=-1)  # (B, 96)

        h = self.full_stem(x_seq)  # (B, 128, 48)
        full_feat = torch.cat([h.mean(dim=-1), h.max(dim=-1)[0]], dim=-1)  # (B, 256)

        tab_e = self.tab_mlp(x_tab)
        emb = self.embed_proj(torch.cat([ring_feat, full_feat, tab_e], dim=-1))
        z_norm = F.normalize(self.supcon_head(emb), dim=-1)
        return self.cls_head(emb), z_norm

class PureClsSupConLoss(nn.Module):
    """
    Replaces broken `reg_head` loss with Cross-Subject Supervised Contrastive Loss + Focal + Soft Macro-F1!
    """
    def __init__(self, num_classes=6, smoothing=0.06, gamma=2.0, f1_w=0.35, supcon_w=0.20, temp=0.12):
        super().__init__()
        self.num_classes = num_classes
        self.smoothing = smoothing
        self.gamma = gamma
        self.f1_w = f1_w
        self.supcon_w = supcon_w
        self.temp = temp

    def forward(self, logits, z_norm, y_cls):
        probs, log_probs = F.softmax(logits, dim=-1), F.log_softmax(logits, dim=-1)
        with torch.no_grad():
            true_dist = torch.full_like(log_probs, self.smoothing / (self.num_classes - 1))
            true_dist.scatter_(1, y_cls.unsqueeze(1), 1.0 - self.smoothing)
        focal = torch.mean(torch.sum(-true_dist * ((1.0 - probs) ** self.gamma) * log_probs, dim=-1))

        one_hot = F.one_hot(y_cls, num_classes=self.num_classes).float()
        tp = torch.sum(probs * one_hot, dim=0)
        fp = torch.sum(probs * (1.0 - one_hot), dim=0)
        fn = torch.sum((1.0 - probs) * one_hot, dim=0)
        soft_f1 = (2.0 * tp + 1e-5) / (2.0 * tp + fp + fn + 1e-5)
        cls_loss = (1.0 - self.f1_w) * focal + self.f1_w * (1.0 - torch.mean(soft_f1))

        # Supervised Contrastive Loss across batch
        sim_mat = torch.matmul(z_norm, z_norm.T) / self.temp
        B = y_cls.size(0)
        eye = torch.eye(B, device=y_cls.device)
        pos_mask = (y_cls.unsqueeze(0) == y_cls.unsqueeze(1)).float() * (1.0 - eye)
        exp_sim = torch.exp(sim_mat - torch.max(sim_mat, dim=1, keepdim=True)[0].detach()) * (1.0 - eye)
        log_prob_sc = sim_mat - torch.max(sim_mat, dim=1, keepdim=True)[0].detach() - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-6)
        pos_cnt = pos_mask.sum(dim=1)
        supcon_loss = -(pos_mask * log_prob_sc).sum(dim=1) / torch.clamp(pos_cnt, min=1.0)
        supcon_loss = supcon_loss[pos_cnt > 0].mean() if (pos_cnt > 0).any() else 0.0

        return cls_loss + self.supcon_w * supcon_loss

def predict_with_tta(model, loader, device):
    model.eval()
    probs_list = []
    with torch.no_grad():
        for batch in loader:
            bx_s, bx_t = batch[0].to(device), batch[1].to(device)
            l0, _ = model(bx_s, bx_t)
            l1, _ = model(torch.roll(bx_s, shifts=-5, dims=-1), bx_t)
            l2, _ = model(torch.roll(bx_s, shifts=5, dims=-1), bx_t)
            p = 0.50 * F.softmax(l0, dim=-1) + 0.25 * F.softmax(l1, dim=-1) + 0.25 * F.softmax(l2, dim=-1)
            probs_list.append(p.cpu().numpy())
    return np.concatenate(probs_list, axis=0)

def train_and_reload_best_swa(model, trn_loader, val_loader, tst_loader, y_val_clean, anchor_submask, criterion, tag):
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1.5e-2)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
    top1_f1, top2_f1 = 0.0, 0.0
    top1_p, top2_p = None, None
    c1, c2 = f"{tag}_top1.pt", f"{tag}_top2.pt"

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for bx_s, bx_t, by_c in trn_loader:
            bx_s, bx_t, by_c = bx_s.to(DEVICE), bx_t.to(DEVICE), by_c.to(DEVICE)
            opt.zero_grad()
            logits, z_norm = model(bx_s, bx_t)
            loss = criterion(logits, z_norm, by_c)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sch.step()

        vp = predict_with_tta(model, val_loader, DEVICE)
        cf1 = f1_score(y_val_clean, np.argmax(vp[anchor_submask], axis=1), average='macro')
        if cf1 > top1_f1:
            top2_f1, top2_p = top1_f1, top1_p
            if os.path.exists(c1):
                os.replace(c1, c2)
            top1_f1, top1_p = cf1, vp
            torch.save(model.state_dict(), c1)
        elif cf1 > top2_f1:
            top2_f1, top2_p = cf1, vp
            torch.save(model.state_dict(), c2)

    if top2_p is not None and os.path.exists(c2):
        val_p = 0.65 * top1_p + 0.35 * top2_p
        model.load_state_dict(torch.load(c1, map_location=DEVICE))
        tp1 = predict_with_tta(model, tst_loader, DEVICE)
        model.load_state_dict(torch.load(c2, map_location=DEVICE))
        tp2 = predict_with_tta(model, tst_loader, DEVICE)
        tst_p = 0.65 * tp1 + 0.35 * tp2
    else:
        val_p = top1_p
        model.load_state_dict(torch.load(c1, map_location=DEVICE))
        tst_p = predict_with_tta(model, tst_loader, DEVICE)

    return val_p, tst_p

def calibrate_session_logits(probs, session_ids, power=0.40, n_iter=4):
    cal_probs = probs.copy()
    for s_id in np.unique(session_ids):
        mask = (session_ids == s_id)
        if np.sum(mask) < 12:
            continue
        P = cal_probs[mask].copy() + 1e-6
        for _ in range(n_iter):
            P = P / np.sum(P, axis=1, keepdims=True)
            P = P * ((1.0 / 6.0) / (np.mean(P, axis=0, keepdims=True) + 1e-6)) ** power
            P = P / np.sum(P, axis=1, keepdims=True)
        cal_probs[mask] = P
    return cal_probs

# =====================================================================
# 4. 5-FOLD TRAINING EXECUTION (3 DEEP NETS + 2 PRUNED GBDTS)
# =====================================================================
(X_seq, X_tab, y_cls, groups, is_anchor,
 X_test_seq, X_test_tab, test_ids, test_sessions) = load_v20_datasets(DATA_DIR)

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS)
oof_nn1 = np.zeros((len(X_seq), 6), np.float32)
oof_nn2 = np.zeros((len(X_seq), 6), np.float32)
oof_nn3 = np.zeros((len(X_seq), 6), np.float32)
oof_lgb = np.zeros((len(X_seq), 6), np.float32)
oof_cat = np.zeros((len(X_seq), 6), np.float32)

test_nn1 = np.zeros((len(X_test_seq), 6), np.float32)
test_nn2 = np.zeros((len(X_test_seq), 6), np.float32)
test_nn3 = np.zeros((len(X_test_seq), 6), np.float32)
test_lgb = np.zeros((len(X_test_seq), 6), np.float32)
test_cat = np.zeros((len(X_test_seq), 6), np.float32)

criterion = PureClsSupConLoss()

for fold, (trn_idx, val_idx) in enumerate(sgkf.split(X_seq, y_cls, groups)):
    print(f"\n==================== FOLD {fold + 1} / {N_FOLDS} ====================")
    clean_val_idx = val_idx[is_anchor[val_idx] == 1]
    anchor_submask = (is_anchor[val_idx] == 1)
    y_val_clean = y_cls[clean_val_idx]

    scaler = StandardScaler()
    X_tab_trn = np.clip(scaler.fit_transform(X_tab[trn_idx]), -8.0, 8.0)
    X_tab_val = np.clip(scaler.transform(X_tab[val_idx]), -8.0, 8.0)
    X_tab_val_clean = np.clip(scaler.transform(X_tab[clean_val_idx]), -8.0, 8.0)
    X_tab_tst = np.clip(scaler.transform(X_test_tab), -8.0, 8.0)

    # Pruned Trees
    scout = LGBMClassifier(n_estimators=220, learning_rate=0.04, max_depth=4, num_leaves=15, subsample=0.8, colsample_bytree=0.6, random_state=42 + fold, n_jobs=-1, verbose=-1)
    scout.fit(X_tab_trn, y_cls[trn_idx])
    keep_cols = np.argsort(scout.feature_importances_)[::-1][:int(X_tab.shape[1] * 0.75)]

    lgb_model = LGBMClassifier(
        n_estimators=1600, learning_rate=0.022, max_depth=5, num_leaves=22, min_child_samples=28,
        min_split_gain=0.01, subsample=0.80, subsample_freq=1, colsample_bytree=0.60,
        reg_alpha=2.0, reg_lambda=4.0, random_state=100 + fold, n_jobs=-1, verbose=-1
    )
    lgb_model.fit(X_tab_trn[:, keep_cols], y_cls[trn_idx], eval_set=[(X_tab_val_clean[:, keep_cols], y_val_clean)], callbacks=[early_stopping(100, verbose=False)])
    oof_lgb[val_idx] = lgb_model.predict_proba(X_tab_val[:, keep_cols])
    fold_tst_lgb = lgb_model.predict_proba(X_tab_tst[:, keep_cols])
    test_lgb += fold_tst_lgb / N_FOLDS

    if HAS_CATBOOST:
        cat_model = CatBoostClassifier(iterations=1600, learning_rate=0.032, depth=5, l2_leaf_reg=6.0, random_seed=100 + fold, task_type='CPU', verbose=False)
        cat_model.fit(X_tab_trn[:, keep_cols], y_cls[trn_idx], eval_set=(X_tab_val_clean[:, keep_cols], y_val_clean), early_stopping_rounds=100, verbose=False)
        oof_cat[val_idx] = cat_model.predict_proba(X_tab_val[:, keep_cols])
        test_cat += cat_model.predict_proba(X_tab_tst[:, keep_cols]) / N_FOLDS
    else:
        oof_cat[val_idx] = oof_lgb[val_idx].copy()
        test_cat += fold_tst_lgb / N_FOLDS

    # 3 Pure-Classification Neural Networks trained with Cross-Subject SupConLoss
    trn_ds = AuraPureClsDataset(X_seq[trn_idx], X_tab_trn, y_cls[trn_idx], is_train=True)
    val_ds = AuraPureClsDataset(X_seq[val_idx], X_tab_val, is_train=False)
    tst_ds = AuraPureClsDataset(X_test_seq, X_tab_tst, is_train=False)

    trn_loader = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    tst_loader = DataLoader(tst_ds, batch_size=BATCH_SIZE, shuffle=False)

    m1 = AuraCrossAttnConformer(in_channels=X_seq.shape[1], tab_dim=X_tab.shape[1]).to(DEVICE)
    vp1, tp1 = train_and_reload_best_swa(m1, trn_loader, val_loader, tst_loader, y_val_clean, anchor_submask, criterion, f"v20_nn1_f{fold}")
    oof_nn1[val_idx] = vp1; test_nn1 += tp1 / N_FOLDS

    m2 = InceptionWaveBiGRUNet(in_channels=X_seq.shape[1], tab_dim=X_tab.shape[1]).to(DEVICE)
    vp2, tp2 = train_and_reload_best_swa(m2, trn_loader, val_loader, tst_loader, y_val_clean, anchor_submask, criterion, f"v20_nn2_f{fold}")
    oof_nn2[val_idx] = vp2; test_nn2 += tp2 / N_FOLDS

    m3 = CircularForearmResNet(in_channels=X_seq.shape[1], tab_dim=X_tab.shape[1]).to(DEVICE)
    vp3, tp3 = train_and_reload_best_swa(m3, trn_loader, val_loader, tst_loader, y_val_clean, anchor_submask, criterion, f"v20_nn3_f{fold}")
    oof_nn3[val_idx] = vp3; test_nn3 += tp3 / N_FOLDS

    oof_nn_fold = (vp1 + vp2 + vp3) / 3.0
    print(
        f"  Fold {fold+1} Clean Val F1 -> NN1(Conformer): {f1_score(y_val_clean, np.argmax(vp1[anchor_submask], 1), average='macro'):.4f} | "
        f"NN2(Inception): {f1_score(y_val_clean, np.argmax(vp2[anchor_submask], 1), average='macro'):.4f} | "
        f"NN3(CircularRing): {f1_score(y_val_clean, np.argmax(vp3[anchor_submask], 1), average='macro'):.4f} | "
        f"3-NN Trio: {f1_score(y_val_clean, np.argmax(oof_nn_fold[anchor_submask], 1), average='macro'):.4f} | "
        f"LGBM: {f1_score(y_val_clean, np.argmax(oof_lgb[clean_val_idx], 1), average='macro'):.4f}"
    )

# =====================================================================
# 5. NEURAL-FOCUSED ENSEMBLE BLENDING & SAFE CALIBRATION
# =====================================================================
clean_mask = (is_anchor == 1)
y_clean = y_cls[clean_mask]

# Optimize internal weights of the 3 Neural Networks first
best_nn_f1, best_nn_w = 0.0, (0.34, 0.33, 0.33)
for a1 in np.linspace(0.20, 0.50, 7):
    for a2 in np.linspace(0.20, 0.50, 7):
        a3 = max(0.0, 1.0 - a1 - a2)
        if a1 + a2 > 1.0:
            continue
        nn_c = a1 * oof_nn1[clean_mask] + a2 * oof_nn2[clean_mask] + a3 * oof_nn3[clean_mask]
        s = f1_score(y_clean, np.argmax(nn_c, axis=1), average='macro')
        if s > best_nn_f1:
            best_nn_f1, best_nn_w = s, (a1, a2, a3)

oof_nn_trio = best_nn_w[0] * oof_nn1 + best_nn_w[1] * oof_nn2 + best_nn_w[2] * oof_nn3
test_nn_trio = best_nn_w[0] * test_nn1 + best_nn_w[1] * test_nn2 + best_nn_w[2] * test_nn3

oof_trees = 0.55 * oof_lgb + 0.45 * oof_cat
test_trees = 0.55 * test_lgb + 0.45 * test_cat

best_raw_f1, best_nn_ratio = 0.0, 0.65
for w_nn in np.linspace(0.40, 0.85, 19):
    blend = w_nn * oof_nn_trio[clean_mask] + (1.0 - w_nn) * oof_trees[clean_mask]
    s = f1_score(y_clean, np.argmax(blend, axis=1), average='macro')
    if s > best_raw_f1:
        best_raw_f1, best_nn_ratio = s, w_nn

oof_blend = best_nn_ratio * oof_nn_trio + (1.0 - best_nn_ratio) * oof_trees
test_blend = best_nn_ratio * test_nn_trio + (1.0 - best_nn_ratio) * test_trees

best_cal_f1, best_power, oof_cal = best_raw_f1, 0.0, oof_blend.copy()
for pwr in [0.20, 0.30, 0.40, 0.50]:
    cand = calibrate_session_logits(oof_blend, groups, power=pwr, n_iter=4)
    s = f1_score(y_clean, np.argmax(cand[clean_mask], axis=1), average='macro')
    if s > best_cal_f1:
        best_cal_f1, best_power, oof_cal = s, pwr, cand

test_cal = calibrate_session_logits(test_blend, test_sessions, power=best_power, n_iter=4) if best_power > 0.0 else test_blend

class_mult = np.ones(6, dtype=np.float32)
tuned_f1 = best_cal_f1
for _ in range(3):
    for c in range(6):
        for m in np.linspace(0.92, 1.08, 17):
            cand = class_mult.copy(); cand[c] = m
            s = f1_score(y_clean, np.argmax(oof_cal[clean_mask] * cand[None, :], axis=1), average='macro')
            if s > tuned_f1:
                tuned_f1, class_mult = s, cand

print("\n========================================================")
print(f"Clean OOF Macro-F1 (3-Neural-Net Trio Alone): {best_nn_f1:.4f} | NN Weights: {np.round(best_nn_w, 2)}")
print(f"Clean OOF Macro-F1 (Neural + Tree Blend):     {best_raw_f1:.4f} | Neural Share: {best_nn_ratio:.2f}")
print(f"Clean OOF Macro-F1 (+Session Calibration):    {best_cal_f1:.4f} | Power: {best_power}")
print(f"Clean OOF Macro-F1 (+Safe Tuned Final):       {tuned_f1:.4f} | Mult: {np.round(class_mult, 3)}")
print("========================================================")
print(classification_report(y_clean, np.argmax(oof_cal[clean_mask] * class_mult[None, :], axis=1), target_names=GESTURES))

final_labels = [INV_GESTURE_MAP[i] for i in np.argmax(test_cal * class_mult[None, :], axis=1)]
sample_sub_files = glob.glob(os.path.join(DATA_DIR, "**", "sample_submission*.csv"), recursive=True)
if sample_sub_files:
    sub = pd.read_csv(sample_sub_files[0])
    id_col, target_col = sub.columns[0], sub.columns[1]
    if len(sub) == len(final_labels) and len(set(test_ids)) == len(test_ids):
        sub[target_col] = sub[id_col].map(dict(zip(test_ids, final_labels))).fillna(pd.Series(final_labels))
    else:
        sub[target_col] = final_labels
else:
    sub = pd.DataFrame({'action_id': test_ids, 'gesture': final_labels})

sub.to_csv("submission.csv", index=False)
sub.to_csv("/kaggle/working/submission.csv", index=False)

with open("submission.csv", "rb") as f:
    sub_hash = hashlib.sha256(f.read()).hexdigest()[:12]

print(f"=== DONE: V20-NEURALFOCUS SUBMISSION READY (SHA256: {sub_hash}) ===")
print(sub.iloc[:, 1].value_counts())
# === END OF COMPLETE V20-NEURALFOCUS SCRIPT ===